# LeetCode 1347 — Minimum Number of Steps to Make Two Strings Anagram bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-steps-to-make-two-strings-anagram/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def minSteps(self, s: str, t: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu khái niệm **anagram** (hoán vị chữ cái) và vì sao "là anagram" ⇔ "**bảng tần suất** chữ cái giống nhau".
2. Biến bài toán "số bước thay ký tự tối thiểu" thành **đếm phần dư tần suất**: `Σ max(0, cnt_s[c] − cnt_t[c])`.
3. Đi từ brute force `O(n²)` (ghép cặp từng ký tự) → sắp xếp + hai con trỏ `O(n log n)` → đếm `O(n)` với `Counter` và **mảng 26 phần tử**.
4. Làm quen các công cụ: `collections.Counter`, phép trừ `Counter`, `ord(c) - ord('a')`.
5. Tránh lỗi: cộng cả hai chiều chênh lệch (đếm gấp đôi), dùng `set`, so sánh theo vị trí, `len()` thay vì `sum()` trên Counter.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s` và `t` **cùng độ dài**. Mỗi bước, bạn được chọn **một ký tự bất kỳ của `t`** và thay nó bằng **một ký tự khác**. Trả về **số bước tối thiểu** để `t` trở thành một **anagram** của `s` (cùng các ký tự với cùng số lần xuất hiện, thứ tự tùy ý).

**Ví dụ 1:** `s = "bab", t = "aba"` → `1` (thay một `'a'` trong `t` bằng `'b'`: `t = "bba"`).

**Ví dụ 2:** `s = "leetcode", t = "practice"` → `5` (thay `'p','r','a','i','c'`).

**Ví dụ 3:** `s = "anagram", t = "mangaar"` → `0` (đã là anagram).

**Ràng buộc:**
- `1 <= s.length <= 5·10^4`
- `s.length == t.length`
- `s`, `t` chỉ gồm chữ cái tiếng Anh viết thường.

In [1]:
import random
import string
import time
from collections import Counter

examples = [
    ("bab", "aba", 1),
    ("leetcode", "practice", 5),
    ("anagram", "mangaar", 0),
]
for s, t, want in examples:
    print(f"s={s!r:12} t={t!r:12} -> mong đợi {want}")

s='bab'        t='aba'        -> mong đợi 1
s='leetcode'   t='practice'   -> mong đợi 5
s='anagram'    t='mangaar'    -> mong đợi 0


## 3. Kiến thức nền

### 3.1 Anagram và sắp xếp

Hai chuỗi là anagram của nhau ⇔ **sắp xếp** các ký tự thì ra cùng một chuỗi. `sorted("bab")` trả về list `['a','b','b']`.

In [2]:
print(sorted("anagram"), sorted("mangaar"))
print("anagram?", sorted("anagram") == sorted("mangaar"))
print("anagram?", sorted("bab") == sorted("aba"))

['a', 'a', 'a', 'g', 'm', 'n', 'r'] ['a', 'a', 'a', 'g', 'm', 'n', 'r']
anagram? True
anagram? False


### 3.2 `collections.Counter` — bảng tần suất

`Counter(s)` là một `dict` đặc biệt: khóa là ký tự, giá trị là số lần xuất hiện. Truy cập khóa không có trả về `0` (không lỗi).

In [3]:
cs = Counter("leetcode")
print(cs)
print("cs['e'] =", cs['e'], "| cs['z'] =", cs['z'])
print("anagram?", Counter("anagram") == Counter("mangaar"))

Counter({'e': 3, 'l': 1, 't': 1, 'c': 1, 'o': 1, 'd': 1})
cs['e'] = 3 | cs['z'] = 0
anagram? True


### 3.3 Phép trừ `Counter` — chỉ giữ phần dương

`Counter(a) - Counter(b)` trừ từng khóa và **loại bỏ các giá trị ≤ 0**. Đây chính là "những ký tự `a` còn thừa sau khi ghép với `b`". Dùng `sum(x.values())` (hoặc `x.total()` từ Python 3.10) để tính tổng.

In [4]:
a, b = Counter("leetcode"), Counter("practice")
print("a - b =", a - b, "-> tổng =", sum((a - b).values()))
print("b - a =", b - a, "-> tổng =", sum((b - a).values()))
print("Số khóa (len) của a - b:", len(a - b), "(KHÔNG phải đáp án!)")

a - b = Counter({'e': 2, 'l': 1, 'o': 1, 'd': 1}) -> tổng = 5
b - a = Counter({'p': 1, 'r': 1, 'a': 1, 'c': 1, 'i': 1}) -> tổng = 5
Số khóa (len) của a - b: 4 (KHÔNG phải đáp án!)


### 3.4 Mảng đếm 26 phần tử với `ord`

Vì chỉ có chữ thường `'a'..'z'`, ta dùng list 26 số nguyên; chỉ số của ký tự `c` là `ord(c) - ord('a')`. Nhanh và gọn hơn dict.

In [5]:
cnt = [0] * 26
for c in "bab":
    cnt[ord(c) - ord('a')] += 1
print(cnt[:5], "(a, b, c, d, e)")
print("ord('a') =", ord('a'), "| ord('b') - ord('a') =", ord('b') - ord('a'))
print("chr(97 + 2) =", chr(97 + 2))

[1, 2, 0, 0, 0] (a, b, c, d, e)
ord('a') = 97 | ord('b') - ord('a') = 1
chr(97 + 2) = c


### 3.5 `max(0, x)` và `sum` trên generator

`sum(max(0, x) for x in lst)` cộng các phần dương. Hữu ích khi một mảng chứa hiệu `cnt_s − cnt_t`.

In [6]:
diff = [2, -1, 0, -1, 3]
print("tổng phần dương:", sum(max(0, x) for x in diff))
print("tổng phần âm (trị tuyệt đối):", sum(max(0, -x) for x in diff))

tổng phần dương: 5
tổng phần âm (trị tuyệt đối): 2


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 2**: `s = "leetcode"`, `t = "practice"`.

| Ký tự | trong s | trong t | s thừa (s − t > 0) |
|---|---|---|---|
| l | 1 | 0 | 1 |
| e | 3 | 1 | 2 |
| t | 1 | 1 | 0 |
| c | 1 | 2 | 0 |
| o | 1 | 0 | 1 |
| d | 1 | 0 | 1 |
| p, r, a, i | 0 | 1 mỗi chữ | 0 |

Tổng phần thừa của `s` = `1 + 2 + 1 + 1 = 5` ✔.

**Trực giác:** ký tự nào của `t` "ghép được" với một ký tự giống nó trong `s` thì giữ lại; các ký tự còn lại của `t` (không ghép được) phải thay. Số ký tự `t` không ghép được = số ký tự `s` không được ghép (cùng độ dài!) = tổng phần thừa.

In [7]:
s, t = "leetcode", "practice"
cs, ct = Counter(s), Counter(t)
for ch in sorted(set(s) | set(t)):
    print(f"{ch}: s={cs[ch]} t={ct[ch]} thừa={max(0, cs[ch] - ct[ch])}")
print("Tổng thừa =", sum(max(0, cs[ch] - ct[ch]) for ch in set(s)))

a: s=0 t=1 thừa=0
c: s=1 t=2 thừa=0
d: s=1 t=0 thừa=1
e: s=3 t=1 thừa=2
i: s=0 t=1 thừa=0
l: s=1 t=0 thừa=1
o: s=1 t=0 thừa=1
p: s=0 t=1 thừa=0
r: s=0 t=1 thừa=0
t: s=1 t=1 thừa=0
Tổng thừa = 5


## 5. Cách 1 — Brute Force / baseline: ghép cặp từng ký tự

**Ý tưởng:** giữ một list `rest` các ký tự **chưa được ghép** của `s`. Duyệt từng ký tự `c` của `t`: nếu `c` có trong `rest` thì ghép (xóa một bản `c` khỏi `rest`); nếu không, ký tự này phải bị thay ⇒ `steps += 1`.

In [8]:
def min_steps_brute(s, t):
    rest = list(s)          # ký tự s chưa được ghép
    steps = 0
    for c in t:
        if c in rest:       # O(n) để tìm
            rest.remove(c)  # O(n) để xóa
        else:
            steps += 1
    return steps

for s, t, want in examples:
    print(f"{s!r:12} {t!r:12} -> {min_steps_brute(s, t)} (mong đợi {want})")

'bab'        'aba'        -> 1 (mong đợi 1)
'leetcode'   'practice'   -> 5 (mong đợi 5)
'anagram'    'mangaar'    -> 0 (mong đợi 0)


**Độ phức tạp:** mỗi `in` và `remove` trên list tốn `O(n)` ⇒ tổng `O(n²)`. Với `n = 5·10^4` là ~2,5·10^9 phép so sánh trong trường hợp xấu — quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** ta lặp lại việc **tìm kiếm** trong list. Nhưng thứ tự ký tự không quan trọng — chỉ **số lượng** của mỗi chữ cái là quan trọng. ⇒ Đếm thay vì tìm.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** "anagram" chỉ quan tâm **tần suất**. Gọi `cs[c]`, `ct[c]` là số lần chữ `c` xuất hiện trong `s`, `t`.

- Với mỗi chữ `c`, tối đa `min(cs[c], ct[c])` ký tự của `t` được **giữ nguyên**.
- Nếu `ct[c] > cs[c]` thì `t` thừa `ct[c] − cs[c]` chữ `c` ⇒ những chữ này **bắt buộc** phải thay.
- Đáp án = `Σ max(0, ct[c] − cs[c])` = `Σ max(0, cs[c] − ct[c])` (hai tổng bằng nhau vì `|s| = |t|`).

**Quy tắc nếu… thì…:**
- Nếu `cs[c] − ct[c] > 0` thì cộng phần đó vào đáp án (thiếu chữ `c` trong `t`, cần tạo ra bằng cách thay).
- Nếu `≤ 0` thì bỏ qua (thừa ở `t` — chính những ký tự đó sẽ bị thay để lấp chỗ thiếu).

Tương đương: đáp án = `n − Σ min(cs[c], ct[c])` (độ dài trừ số ký tự ghép được).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + hai con trỏ

**Giải thích:** sắp xếp `s` và `t`. Hai con trỏ `i`, `j` chạy song song như phép trộn (merge): nếu `a[i] == b[j]` thì ghép được (tăng cả hai, `matched += 1`); nếu `a[i] < b[j]` thì tăng `i`; ngược lại tăng `j`. Đáp án `n − matched`.

In [9]:
def min_steps_sort(s, t):
    a, b = sorted(s), sorted(t)
    i = j = matched = 0
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            matched += 1
            i += 1
            j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return len(s) - matched

for s, t, want in examples:
    print(f"{s!r:12} {t!r:12} -> {min_steps_sort(s, t)} (mong đợi {want})")

'bab'        'aba'        -> 1 (mong đợi 1)
'leetcode'   'practice'   -> 5 (mong đợi 5)
'anagram'    'mangaar'    -> 0 (mong đợi 0)


Code nộp LeetCode (Cách 2):

In [10]:
class Solution:
    def minSteps(self, s: str, t: str) -> int:
        a, b = sorted(s), sorted(t)
        i = j = matched = 0
        while i < len(a) and j < len(b):
            if a[i] == b[j]:
                matched += 1; i += 1; j += 1
            elif a[i] < b[j]:
                i += 1
            else:
                j += 1
        return len(s) - matched

print(Solution().minSteps("leetcode", "practice"))

5


**Độ phức tạp:** `O(n log n)` cho sắp xếp, `O(n)` cho hai con trỏ; bộ nhớ `O(n)` cho hai list đã sắp xếp.

### 7.2 Cách 3 — `Counter` và phép trừ

**Giải thích:** `Counter(s) - Counter(t)` giữ lại đúng phần `s` thừa so với `t` (giá trị dương). Tổng các giá trị là đáp án. Một dòng!

In [11]:
def min_steps_counter(s, t):
    return sum((Counter(s) - Counter(t)).values())

def min_steps_counter_min(s, t):
    """Biến thể: n - số ký tự ghép được."""
    cs, ct = Counter(s), Counter(t)
    return len(s) - sum(min(cs[c], ct[c]) for c in cs)

for s, t, want in examples:
    print(f"{s!r:12} {t!r:12} -> {min_steps_counter(s, t)}, {min_steps_counter_min(s, t)} (mong đợi {want})")

'bab'        'aba'        -> 1, 1 (mong đợi 1)
'leetcode'   'practice'   -> 5, 5 (mong đợi 5)
'anagram'    'mangaar'    -> 0, 0 (mong đợi 0)


Code nộp LeetCode (Cách 3):

In [12]:
class Solution:
    def minSteps(self, s: str, t: str) -> int:
        return sum((Counter(s) - Counter(t)).values())

print(Solution().minSteps("bab", "aba"))

1


**Độ phức tạp:** `O(n)` để đếm; phép trừ `O(Σ)` với `Σ ≤ 26`. Bộ nhớ `O(Σ)`.

### 7.3 Cách 4 — Một mảng đếm 26 phần tử (+1 / −1)

**Giải thích từng đoạn:**
1. `cnt = [0]*26`.
2. Duyệt song song `s[i]`, `t[i]`: `cnt[s[i]] += 1`, `cnt[t[i]] -= 1`. Sau vòng lặp `cnt[c] = cs[c] − ct[c]`.
3. Đáp án = tổng các phần tử dương.

Chỉ một mảng, một vòng lặp, không cấp phát dict — cách "chuẩn" trong phỏng vấn.

In [13]:
def min_steps_array(s, t):
    cnt = [0] * 26
    for a, b in zip(s, t):
        cnt[ord(a) - 97] += 1
        cnt[ord(b) - 97] -= 1
    return sum(x for x in cnt if x > 0)

for s, t, want in examples:
    print(f"{s!r:12} {t!r:12} -> {min_steps_array(s, t)} (mong đợi {want})")

'bab'        'aba'        -> 1 (mong đợi 1)
'leetcode'   'practice'   -> 5 (mong đợi 5)
'anagram'    'mangaar'    -> 0 (mong đợi 0)


Code nộp LeetCode (Cách 4):

In [14]:
class Solution:
    def minSteps(self, s: str, t: str) -> int:
        cnt = [0] * 26
        for a, b in zip(s, t):
            cnt[ord(a) - 97] += 1
            cnt[ord(b) - 97] -= 1
        return sum(x for x in cnt if x > 0)

print(Solution().minSteps("anagram", "mangaar"))

0


**Độ phức tạp:** `O(n + 26)` thời gian, `O(26) = O(1)` bộ nhớ.

### 7.4 Cách 5 — `str.count` cho từng chữ cái

Duyệt 26 chữ cái, mỗi chữ gọi `s.count(c)` và `t.count(c)` (được cài bằng C nên rất nhanh). Tổng `26 × 2` lần quét chuỗi ⇒ `O(26·n)`, vẫn tuyến tính.

In [15]:
def min_steps_count(s, t):
    return sum(max(0, s.count(c) - t.count(c)) for c in string.ascii_lowercase)

for s, t, want in examples:
    print(f"{s!r:12} {t!r:12} -> {min_steps_count(s, t)} (mong đợi {want})")

'bab'        'aba'        -> 1 (mong đợi 1)
'leetcode'   'practice'   -> 5 (mong đợi 5)
'anagram'    'mangaar'    -> 0 (mong đợi 0)


## 8. TRACE từng bước

Trace Cách 4 trên `s = "bab"`, `t = "aba"` (chỉ in các ô `a`, `b`):

| i | s[i] | t[i] | cnt[a] | cnt[b] |
|---|---|---|---|---|
| 0 | b | a | −1 | 1 |
| 1 | a | b | 0 | 0 |
| 2 | b | a | −1 | 1 |

Tổng phần dương = `cnt[b] = 1` ⇒ đáp án **1** ✔ (`s` thiếu ở `t` đúng một chữ `b`, `t` thừa đúng một chữ `a`).

In [16]:
def trace_array(s, t):
    cnt = [0] * 26
    letters = sorted(set(s) | set(t))
    print("i | s[i] t[i] | " + " ".join(f"{c:>3}" for c in letters))
    for i, (a, b) in enumerate(zip(s, t)):
        cnt[ord(a) - 97] += 1
        cnt[ord(b) - 97] -= 1
        print(f"{i} |  {a}    {b}  | " + " ".join(f"{cnt[ord(c) - 97]:>3}" for c in letters))
    ans = sum(x for x in cnt if x > 0)
    print("phần dương:", {chr(97 + k): x for k, x in enumerate(cnt) if x > 0}, "-> đáp án", ans)
    return ans

trace_array("bab", "aba")
print()
trace_array("leetcode", "practice")

i | s[i] t[i] |   a   b
0 |  b    a  |  -1   1
1 |  a    b  |   0   0
2 |  b    a  |  -1   1
phần dương: {'b': 1} -> đáp án 1

i | s[i] t[i] |   a   c   d   e   i   l   o   p   r   t
0 |  l    p  |   0   0   0   0   0   1   0  -1   0   0
1 |  e    r  |   0   0   0   1   0   1   0  -1  -1   0
2 |  e    a  |  -1   0   0   2   0   1   0  -1  -1   0
3 |  t    c  |  -1  -1   0   2   0   1   0  -1  -1   1
4 |  c    t  |  -1   0   0   2   0   1   0  -1  -1   0
5 |  o    i  |  -1   0   0   2  -1   1   1  -1  -1   0
6 |  d    c  |  -1  -1   1   2  -1   1   1  -1  -1   0
7 |  e    e  |  -1  -1   1   2  -1   1   1  -1  -1   0
phần dương: {'d': 1, 'e': 2, 'l': 1, 'o': 1} -> đáp án 5


5

Trace Cách 2 (hai con trỏ) trên `"leetcode"` / `"practice"`:

In [17]:
def trace_sort(s, t):
    a, b = sorted(s), sorted(t)
    print("a =", "".join(a), "| b =", "".join(b))
    i = j = matched = 0
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            matched += 1
            act = f"ghép '{a[i]}'"
            i += 1; j += 1
        elif a[i] < b[j]:
            act = f"'{a[i]}' < '{b[j]}' -> i++"
            i += 1
        else:
            act = f"'{a[i]}' > '{b[j]}' -> j++"
            j += 1
        print(f"  {act:22} | i={i} j={j} matched={matched}")
    print("đáp án = n - matched =", len(s) - matched)

trace_sort("leetcode", "practice")

a = cdeeelot | b = acceiprt
  'c' > 'a' -> j++       | i=0 j=1 matched=0
  ghép 'c'               | i=1 j=2 matched=1
  'd' > 'c' -> j++       | i=1 j=3 matched=1
  'd' < 'e' -> i++       | i=2 j=3 matched=1
  ghép 'e'               | i=3 j=4 matched=2
  'e' < 'i' -> i++       | i=4 j=4 matched=2
  'e' < 'i' -> i++       | i=5 j=4 matched=2
  'l' > 'i' -> j++       | i=5 j=5 matched=2
  'l' < 'p' -> i++       | i=6 j=5 matched=2
  'o' < 'p' -> i++       | i=7 j=5 matched=2
  't' > 'p' -> j++       | i=7 j=6 matched=2
  't' > 'r' -> j++       | i=7 j=7 matched=2
  ghép 't'               | i=8 j=8 matched=3
đáp án = n - matched = 5


## 9. Code nộp LeetCode

In [18]:
class Solution:
    def minSteps(self, s: str, t: str) -> int:
        cnt = [0] * 26                 # cnt[c] = (số c trong s) - (số c trong t)
        for a, b in zip(s, t):
            cnt[ord(a) - 97] += 1
            cnt[ord(b) - 97] -= 1
        return sum(x for x in cnt if x > 0)

sol = Solution()
for s, t, want in examples:
    assert sol.minSteps(s, t) == want
    print(f"minSteps({s!r}, {t!r}) = {sol.minSteps(s, t)}")

minSteps('bab', 'aba') = 1
minSteps('leetcode', 'practice') = 5
minSteps('anagram', 'mangaar') = 0


## 10. Vì sao thuật toán đúng

Gọi `D = Σ_c max(0, cs[c] − ct[c])`.

**Cận dưới (cần ít nhất `D` bước):** sau khi biến đổi, `t` phải có đúng `cs[c]` chữ `c`. Một bước thay chỉ tạo thêm **tối đa một** chữ mới. Với mỗi `c` mà `ct[c] < cs[c]`, cần tạo thêm `cs[c] − ct[c]` chữ `c` ⇒ tổng cộng ít nhất `D` bước.

**Đủ (`D` bước là làm được):** vì `|s| = |t|` nên `Σ(cs[c] − ct[c]) = 0` ⇒ tổng phần thừa của `t` (các `c` có `ct[c] > cs[c]`) cũng bằng `D`. Lấy đúng `D` ký tự thừa đó, thay mỗi cái bằng một chữ đang thiếu ⇒ sau `D` bước tần suất khớp hoàn toàn.

Vậy đáp án đúng bằng `D`. Cách 2 tính `n − matched`, trong đó `matched = Σ min(cs[c], ct[c])` (hai con trỏ ghép tối đa theo từng chữ) ⇒ `n − Σ min = Σ (cs[c] − min) = D`.

## 11. Phân tích Time / Space

Gọi `n = |s| = |t|`, `Σ = 26` (kích thước bảng chữ cái).

| Cách | Time | Space |
|---|---|---|
| 1. Ghép cặp bằng list | `O(n²)` | `O(n)` |
| 2. Sắp xếp + hai con trỏ | `O(n log n)` | `O(n)` |
| 3. Counter trừ | `O(n + Σ)` | `O(Σ)` |
| 4. Mảng 26 | `O(n + Σ)` | `O(Σ)` = `O(1)` |
| 5. `str.count` × 26 | `O(Σ·n)` | `O(1)` |

Cách 1 tốt nhất khi `s`, `t` gần như là anagram và ký tự cần tìm nằm đầu list; xấu nhất khi phải quét hết list mỗi lần.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
METHODS = {
    "brute": min_steps_brute,
    "sort": min_steps_sort,
    "counter": min_steps_counter,
    "counter_min": min_steps_counter_min,
    "array": min_steps_array,
    "count": min_steps_count,
    "submit": Solution().minSteps,
}
cases = examples + [
    ("a", "a", 0),                 # n = 1, giống nhau
    ("a", "b", 1),                 # n = 1, khác nhau
    ("abc", "def", 3),             # không chung ký tự nào
    ("aaaa", "bbbb", 4),
    ("aabb", "bbaa", 0),           # hoán vị
    ("aaab", "abbb", 2),
    ("friend", "family", 4),
    ("abcdefghijklmnopqrstuvwxyz", "zyxwvutsrqponmlkjihgfedcba", 0),
]
for s, t, want in cases:
    for name, f in METHODS.items():
        got = f(s, t)
        assert got == want, (name, s, t, got, want)
print(f"{len(cases)} ca × {len(METHODS)} cách: đúng hết")

11 ca × 7 cách: đúng hết


### 12.2 Đối chiếu ngẫu nhiên (seed cố định) + tính chất đối xứng

Tính chất: `minSteps(s, t) == minSteps(t, s)` (vì cùng độ dài), và `0 ≤ đáp án ≤ n`.

In [20]:
rng = random.Random(1347)
for _ in range(800):
    n = rng.randint(1, 30)
    alpha = string.ascii_lowercase[:rng.randint(1, 26)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    t = "".join(rng.choice(alpha) for _ in range(n))
    ref = min_steps_brute(s, t)
    for name, f in METHODS.items():
        assert f(s, t) == ref, (name, s, t)
    assert min_steps_array(t, s) == ref
    assert 0 <= ref <= n
print("800 bộ ngẫu nhiên: mọi cách khớp brute force, đối xứng đúng")

800 bộ ngẫu nhiên: mọi cách khớp brute force, đối xứng đúng


### 12.3 Kích thước lớn nhất `n = 5·10^4`

In [21]:
rng = random.Random(7)
n = 5 * 10**4
s = "".join(rng.choice(string.ascii_lowercase) for _ in range(n))
t = "".join(rng.choice(string.ascii_lowercase) for _ in range(n))
for name in ["sort", "counter", "array", "count"]:
    st = time.perf_counter()
    r = METHODS[name](s, t)
    print(f"{name:8} -> {r}  ({time.perf_counter() - st:.4f}s)")
assert len({METHODS[k](s, t) for k in ["sort", "counter", "array", "count"]}) == 1

sort     -> 492  (0.0199s)
counter  -> 492  (0.0046s)


array    -> 492  (0.0089s)
count    -> 492  (0.0024s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng trị tuyệt đối chênh lệch của mọi chữ (đếm gấp đôi)

In [22]:
def wrong_abs(s, t):
    cs, ct = Counter(s), Counter(t)
    return sum(abs(cs[c] - ct[c]) for c in set(s) | set(t))

for s, t, want in examples:
    print(f"{s!r:12} {t!r:12} -> sai cho {wrong_abs(s, t)}, đúng là {want}")

'bab'        'aba'        -> sai cho 2, đúng là 1
'leetcode'   'practice'   -> sai cho 10, đúng là 5
'anagram'    'mangaar'    -> sai cho 0, đúng là 0


**Vì sao sai:** mỗi bước thay vừa **bớt một** chữ thừa vừa **thêm một** chữ thiếu ⇒ một bước sửa được **2** đơn vị chênh lệch. Tổng `|chênh|` gấp đôi đáp án. **Sửa:** chỉ cộng phần dương (hoặc chia 2).

### Lỗi 2 — So sánh theo vị trí

In [23]:
def wrong_position(s, t):
    return sum(a != b for a, b in zip(s, t))

for s, t, want in examples:
    print(f"{s!r:12} {t!r:12} -> sai cho {wrong_position(s, t)}, đúng là {want}")

'bab'        'aba'        -> sai cho 3, đúng là 1
'leetcode'   'practice'   -> sai cho 7, đúng là 5
'anagram'    'mangaar'    -> sai cho 5, đúng là 0


**Vì sao sai:** anagram cho phép **đổi thứ tự** tùy ý, nên `"anagram"` và `"mangaar"` không cần bước nào dù khác nhau ở nhiều vị trí. **Sửa:** so sánh tần suất, không so vị trí.

### Lỗi 3 — Dùng `set` (bỏ mất số lần lặp)

In [24]:
def wrong_set(s, t):
    return len(set(s) - set(t))

for s, t, want in examples + [("aaab", "abbb", 2)]:
    print(f"{s!r:12} {t!r:12} -> sai cho {wrong_set(s, t)}, đúng là {want}")

'bab'        'aba'        -> sai cho 0, đúng là 1
'leetcode'   'practice'   -> sai cho 3, đúng là 5
'anagram'    'mangaar'    -> sai cho 0, đúng là 0
'aaab'       'abbb'       -> sai cho 0, đúng là 2


**Vì sao sai:** `set` chỉ cho biết chữ có xuất hiện hay không. `"bab"` vs `"aba"` có cùng tập `{a, b}` nhưng `t` thiếu một chữ `b`. **Sửa:** dùng `Counter` / mảng đếm.

### Lỗi 4 — `len()` của Counter hiệu thay vì `sum()`

In [25]:
def wrong_len(s, t):
    return len(Counter(s) - Counter(t))     # số KHÓA, không phải tổng

for s, t in [("leetcode", "practice"), ("aaaa", "bbbb")]:
    print(f"{s!r:12} {t!r:12} -> sai cho {wrong_len(s, t)}, đúng là {min_steps_counter(s, t)}")

'leetcode'   'practice'   -> sai cho 4, đúng là 5
'aaaa'       'bbbb'       -> sai cho 1, đúng là 4


**Vì sao sai:** `Counter("aaaa") - Counter("bbbb") = {'a': 4}` có **1 khóa** nhưng cần **4** bước. **Sửa:** `sum((Counter(s) - Counter(t)).values())`.

### Lỗi 5 — Cộng cả phần âm (dùng `sum(cnt)`)

In [26]:
def wrong_sum_all(s, t):
    cnt = [0] * 26
    for a, b in zip(s, t):
        cnt[ord(a) - 97] += 1
        cnt[ord(b) - 97] -= 1
    return sum(cnt)              # luôn bằng 0 vì |s| == |t|

print("sai cho", wrong_sum_all("leetcode", "practice"), "| đúng là", 5)

sai cho 0 | đúng là 5


**Vì sao sai:** tổng mọi phần tử luôn bằng `|s| − |t| = 0`. **Sửa:** chỉ cộng các `x > 0`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy chữ **anagram / hoán vị / "thứ tự tùy ý"** ⇒ nghĩ ngay tới **bảng tần suất** (`Counter` hoặc mảng 26).
- "Số bước thay tối thiểu để hai multiset bằng nhau" (cùng kích thước) ⇒ **tổng phần dương của hiệu tần suất**.
- Chỉ chữ thường ⇒ mảng `[0]*26` với `ord(c) - 97` nhanh hơn dict.
- Một mảng `+1` cho `s`, `−1` cho `t` — kỹ thuật "**một mảng hai chiều đếm**" dùng lại ở 242 (Valid Anagram), 438 (Find All Anagrams), 567 (Permutation in String).
- Kiểm tra nhanh: đáp án ≤ `n`, đối xứng khi đổi `s` và `t`.

## 15. Biến thể của bài

1. **LeetCode 242 — Valid Anagram:** trả về `True/False` ⇔ đáp án bài này bằng 0.
2. **LeetCode 2186 — Minimum Number of Steps to Make Two Strings Anagram II:** được **chèn** ký tự vào `s` hoặc `t` (độ dài khác nhau) ⇒ đáp án `Σ |cs[c] − ct[c]|` (lần này **không** chia 2!).
3. **LeetCode 1657 — Determine if Two Strings Are Close:** được hoán đổi hai ký tự và đổi toàn bộ một chữ thành chữ khác.
4. **Trả về chuỗi `t` mới** nhỏ nhất theo thứ tự từ điển sau đúng số bước tối thiểu (xem Bài 5).
5. **Bảng chữ cái lớn (Unicode)**: dùng `Counter` thay mảng 26.

In [27]:
def min_steps_ii(s, t):
    """Biến thể 2 (LC 2186): được chèn ký tự vào s hoặc t."""
    cs, ct = Counter(s), Counter(t)
    return sum(abs(cs[c] - ct[c]) for c in cs.keys() | ct.keys())

print("LC 2186:", min_steps_ii("leetcode", "coats"), "(mong đợi 7)")
print("LC 2186:", min_steps_ii("night", "thing"), "(mong đợi 0)")

LC 2186: 7 (mong đợi 7)
LC 2186: 0 (mong đợi 0)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ghép cặp list | tìm & xóa từng ký tự | `O(n²)` | `O(n)` | minh họa, n nhỏ, làm trọng tài |
| 2. Sắp xếp + 2 con trỏ | ghép như merge | `O(n log n)` | `O(n)` | khi không được dùng bảng băm |
| 3. Counter trừ | phần dư multiset | `O(n)` | `O(Σ)` | code Python ngắn nhất |
| 4. Mảng 26 (+1/−1) | hiệu tần suất | `O(n)` | `O(1)` | phỏng vấn, tối ưu |
| 5. `str.count` × 26 | đếm từng chữ | `O(26n)` | `O(1)` | nhanh thực tế nhờ C, code 1 dòng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Đoán `min_steps_array("xxyyzz", "xyzxyz")` và `min_steps_array("abcd", "aaaa")`.

**Bài 2 (tính tay):** Lập bảng tần suất và tính đáp án cho `s = "mississippi"`, `t = "pipissimsis"` — rồi so với `s = "mississippi"`, `t = "aaaaaaaaaaa"`.

**Bài 3 (điền TODO):** Hoàn thiện `is_anagram(s, t)` (LeetCode 242) chỉ dùng **một** mảng 26 phần tử và trả về sớm `False` nếu độ dài khác nhau.

**Bài 4 (tự viết lại):** Viết `min_steps_ii(s, t)` (LeetCode 2186, độ dài có thể khác nhau, được chèn ký tự vào cả hai chuỗi) bằng **mảng 26** thay vì Counter.

**Bài 5 (mini challenge):** Viết `smallest_anagram_fix(s, t)` trả về chuỗi `t'` sao cho: `t'` là anagram của `s`, `t'` khác `t` ở **đúng** `minSteps(s, t)` vị trí (số bước tối thiểu), và `t'` **nhỏ nhất theo thứ tự từ điển** trong các chuỗi như vậy.

In [28]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def is_anagram(s, t):
    # TODO: nếu độ dài khác nhau -> False
    # TODO: một mảng cnt 26 phần tử, +1 cho s, -1 cho t
    # TODO: trả về True nếu mọi phần tử bằng 0
    pass

In [29]:
# Bài 4, Bài 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_steps_ii_array(s, t):
    # TODO
    pass

def smallest_anagram_fix(s, t):
    # TODO: xác định các vị trí BẮT BUỘC phải thay (ký tự thừa của t),
    #       rồi chọn chữ thay thế sao cho chuỗi nhỏ nhất theo từ điển
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Lập bảng tần suất. (2) `"xxyyzz"` và `"xyzxyz"` có cùng tần suất. (3) `"abcd"` vs `"aaaa"`: `s` thừa `b, c, d`.

**Bài 2:** (1) `mississippi`: m1 i4 s4 p2. (2) Đếm `pipissimsis` cẩn thận. (3) Với `"aaaaaaaaaaa"`: `s` không có `a` nên phải thay tất cả.

**Bài 3:** (1) `if len(s) != len(t): return False`. (2) `cnt[ord(a)-97] += 1; cnt[ord(b)-97] -= 1`. (3) `return all(x == 0 for x in cnt)`.

**Bài 4:** (1) Hai vòng lặp riêng vì độ dài khác nhau. (2) Đáp án là tổng trị tuyệt đối. (3) Không chia 2 vì chèn một ký tự chỉ sửa được 1 đơn vị chênh lệch.

**Bài 5:** (1) Với mỗi chữ `c`, `t` thừa `ct[c] − cs[c]` lần; để giữ được nhiều nhất, ta chỉ thay đúng số đó. (2) Ký tự thay vào là multiset các chữ thiếu (cố định). Để nhỏ nhất theo từ điển, các chữ thiếu nên đặt **sắp tăng dần** vào các vị trí bị thay theo thứ tự trái → phải. (3) Chọn **vị trí nào** của chữ thừa `c` để thay: duyệt trái → phải, thay một chữ `c` thừa nếu chữ thiếu nhỏ nhất hiện tại `< c` (làm chuỗi nhỏ đi) hoặc nếu số chữ `c` còn lại phía sau không đủ để "bỏ qua"; ngược lại giữ nguyên. Kiểm chứng bằng brute force trên chuỗi ngắn.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
print("Bài 1:", min_steps_array("xxyyzz", "xyzxyz"), min_steps_array("abcd", "aaaa"))
assert min_steps_array("xxyyzz", "xyzxyz") == 0
assert min_steps_array("abcd", "aaaa") == 3

# Bài 2
print("Bài 2:", Counter("mississippi"), Counter("pipissimsis"))
assert min_steps_array("mississippi", "pipissimsis") == 0
assert min_steps_array("mississippi", "aaaaaaaaaaa") == 11

Bài 1: 0 3
Bài 2: Counter({'i': 4, 's': 4, 'p': 2, 'm': 1}) Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})


In [31]:
# Bài 3
def is_anagram_sol(s, t):
    if len(s) != len(t):
        return False
    cnt = [0] * 26
    for a, b in zip(s, t):
        cnt[ord(a) - 97] += 1
        cnt[ord(b) - 97] -= 1
    return all(x == 0 for x in cnt)

assert is_anagram_sol("anagram", "nagaram") and not is_anagram_sol("rat", "car") and not is_anagram_sol("a", "ab")

# Bài 4
def min_steps_ii_array_sol(s, t):
    cnt = [0] * 26
    for a in s:
        cnt[ord(a) - 97] += 1
    for b in t:
        cnt[ord(b) - 97] -= 1
    return sum(abs(x) for x in cnt)

rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 12)))
    t = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 12)))
    assert min_steps_ii_array_sol(s, t) == min_steps_ii(s, t)
    assert is_anagram_sol(s, t) == (sorted(s) == sorted(t))
print("Bài 3, 4: đúng trên 300 bộ ngẫu nhiên")

Bài 3, 4: đúng trên 300 bộ ngẫu nhiên


In [32]:
# Bài 5
from itertools import permutations

def smallest_anagram_fix_sol(s, t):
    cs, ct = Counter(s), Counter(t)
    extra = {c: ct[c] - cs[c] for c in ct if ct[c] > cs[c]}       # số chữ c phải thay
    need = sorted((cs - ct).elements())                          # chữ thiếu, tăng dần
    remaining = Counter(t)                                       # số chữ c còn lại từ vị trí i trở đi
    res, k = [], 0
    for c in t:
        replace = False
        if extra.get(c, 0) > 0:
            # bắt buộc thay nếu số c còn lại (kể cả vị trí này) bằng số c còn phải thay
            if remaining[c] == extra[c] or need[k] < c:
                replace = True
        remaining[c] -= 1
        if replace:
            extra[c] -= 1
            res.append(need[k]); k += 1
        else:
            res.append(c)
    return "".join(res)

def smallest_anagram_fix_brute(s, t):
    best, steps = None, min_steps_array(s, t)
    for p in set(permutations(s)):
        cand = "".join(p)
        if sum(a != b for a, b in zip(cand, t)) == steps and (best is None or cand < best):
            best = cand
    return best

print("Bài 5:", smallest_anagram_fix_sol("bab", "aba"), smallest_anagram_fix_sol("leetcode", "practice"))
rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 6)
    s = "".join(rng.choice("abcd") for _ in range(n))
    t = "".join(rng.choice("abcd") for _ in range(n))
    assert smallest_anagram_fix_sol(s, t) == smallest_anagram_fix_brute(s, t), (s, t)
print("Bài 5: khớp brute force trên 300 bộ ngẫu nhiên")

Bài 5: abb deectloe
Bài 5: khớp brute force trên 300 bộ ngẫu nhiên


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template mảng 26:**

```python
cnt = [0] * 26
for a, b in zip(s, t):
    cnt[ord(a) - 97] += 1      # s
    cnt[ord(b) - 97] -= 1      # t
return sum(x for x in cnt if x > 0)
```

**Template Counter:** `return sum((Counter(s) - Counter(t)).values())`

**Checklist trước khi submit:**
- [ ] Chỉ cộng phần **dương** (không cộng `abs`, không cộng tất cả).
- [ ] Dùng **tần suất**, không so sánh theo vị trí, không dùng `set`.
- [ ] Với Counter: `sum(...values())`, không `len(...)`.
- [ ] Thử `"bab"/"aba"` → 1, `"anagram"/"mangaar"` → 0, `"abc"/"def"` → 3.
- [ ] Độ phức tạp `O(n)`, không `list.remove` trong vòng lặp.
- [ ] Đúng chữ ký: `minSteps(self, s: str, t: str) -> int`.